# Train the model

Awesome, now that the environment is setup training can begin. This tutorial also uses [Stable Baselines3](https://stable-baselines3.readthedocs.io/en/master/), so make sure these dependencies are installed:
```bash
pip install stable-baselines3
pip install gymnasium
```

In [ ]:
import os
from widgets.python.env import ATRVEnv
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

gym.register(
    id="kdFlex-ATRVjr",
    entry_point=ATRVEnv,
    max_episode_steps=200,
)

## Load the Environment

Here, we load our training environment by registering it with gymnasium and wrapping it with DummyVecEnv to be compatible with Stable Baselines3. Here, we demonstrate using a function to create multiple environments in a single vectorized environment. While DummyVecEnv will only batch them in a single process, it is also possible to parallelize across different processes using [SubProcVecEnv](https://stable-baselines.readthedocs.io/en/master/guide/vec_envs.html#subprocvecenv).

There are also 2 other wrappers we use.
- [Monitor](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) tracks key data for training
- [VecNormalize](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) automatically normalizes the observations and reward since training models can be unstable with large or varied values. 

In [2]:
train_env = gym.make("kdFlex-ATRVjr")
train_env = DummyVecEnv([lambda: train_env])
train_env = VecNormalize(train_env)

## Setup Callbacks

A common technique is to use a separate evaluation environment to run periodically during training and stop training if the model hasn't improved for a specified period of time to prevent overfitting. This section accomplishes this by adding an EvalCallback.
- EvalCallback also saves the best model into ./models/best_model.zip

In [3]:
 # Stop training if there is no improvement after more than 20 evaluations
eval_env = gym.make("kdFlex-ATRVjr", max_episode_steps=200)
eval_env = DummyVecEnv([lambda: eval_env])  # Wrap in DummyVecEnv
eval_env = VecNormalize(eval_env, training=False, norm_obs=True, norm_reward=False, clip_obs=10.)
stop_train_callback = StopTrainingOnNoModelImprovement(
    max_no_improvement_evals=30,
    min_evals=10,
    verbose=1
)
eval_callback = EvalCallback(
    eval_env,
    eval_freq=3000,
    best_model_save_path="./logs/",
    log_path="./logs/",
    callback_after_eval=stop_train_callback,
    deterministic=True,
    verbose=1
)

## Train a SAC Model

Because the action space is continuous, a Soft Actor Critic (SAC) model is used with the default hyperparameters.

In [4]:
model = SAC(
    "MlpPolicy",
    train_env,
    tensorboard_log="./tensorboard/",
    verbose=1,
    learning_starts=1000,
)

Using cuda device


In [5]:
# takes around 30 minutes
model.learn(total_timesteps=1000000, callback=eval_callback)

Logging to ./tensorboard/SAC_1
----------------------------
| time/              |     |
|    episodes        | 4   |
|    fps             | 960 |
|    time_elapsed    | 0   |
|    total_timesteps | 800 |
----------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 376      |
|    time_elapsed    | 4        |
|    total_timesteps | 1600     |
| train/             |          |
|    actor_loss      | -5.18    |
|    critic_loss     | 0.0268   |
|    ent_coef        | 0.836    |
|    ent_coef_loss   | -0.603   |
|    learning_rate   | 0.0003   |
|    n_updates       | 599      |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 12       |
|    fps             | 296      |
|    time_elapsed    | 8        |
|    total_timesteps | 2400     |
| train/             |          |
|    actor_loss      | -8.03    |
|    critic_loss  

/home/kane/Documents/code/other/personal_website/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=3000, episode_reward=-197.06 +/- 47.59
Episode length: 200.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 200      |
|    mean_reward     | -197     |
| time/              |          |
|    total_timesteps | 3000     |
| train/             |          |
|    actor_loss      | -9.6     |
|    critic_loss     | 0.285    |
|    ent_coef        | 0.549    |
|    ent_coef_loss   | -2.01    |
|    learning_rate   | 0.0003   |
|    n_updates       | 1999     |
---------------------------------
New best mean reward!
---------------------------------
| time/              |          |
|    episodes        | 20       |
|    fps             | 232      |
|    time_elapsed    | 16       |
|    total_timesteps | 3769     |
| train/             |          |
|    actor_loss      | -11.2    |
|    critic_loss     | 0.0145   |
|    ent_coef        | 0.436    |
|    ent_coef_loss   | -2.8     |
|    learning_rate   | 0.0003   |
|   

## Test the Model
Because the model was trained on normalized observation values, it is important that the test environment also uses normalized values. Therefore, the normalization configuration is first saved into ./models/vec_normalize.pkl before it is loaded into the test environment. 

In [7]:
# save the normalization statistics
train_env.save("./models/vec_normalize.pkl")
train_env.close()

[warning] Object '/home/kane/Documents/code/other/personal_website/widgets/python/other/atrvjr/grid.png (Karana::Core::Base)' with id=861 is not destroyed and is not a zombie.

[warning] Object 'frame_container (Karana::Frame::FrameContainer)' with id=970 is not destroyed and is not a zombie.

[warning] Object 'root (Karana::Frame::Frame)' with id=971 is not destroyed and is not a zombie.

[warning] Object 'phong_material (Karana::Core::Base)' with id=973 is not destroyed and is not a zombie.

[warning] Object 'phong_material (Karana::Core::Base)' with id=974 is not destroyed and is not a zombie.

[warning] Object 'phong_material (Karana::Core::Base)' with id=975 is not destroyed and is not a zombie.

[warning] Object 'phong_material (Karana::Core::Base)' with id=976 is not destroyed and is not a zombie.

[warning] Object 'phong_material (Karana::Core::Base)' with id=977 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=978 is not destro

AssertionError: 

In [8]:
test_env = gym.make("kdFlex-ATRVjr", render_mode="human", sync_real_time=True)
test_env = DummyVecEnv([lambda: test_env])
test_env = VecNormalize.load("./models/vec_normalize.pkl", test_env)
test_env.training = False
test_env.norm_reward = False

[WebUI] Listening at http://localhost:38779


In [11]:
terminated = False
obs = test_env.reset()
while not terminated:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, info = test_env.step(action)

A working model is also saved inside ./models/demo_models if you have difficulty training the model.

In [8]:
test_env.close()
test_env = gym.make("kdFlex-ATRVjr", render_mode="human", sync_real_time=True)
test_env = DummyVecEnv([lambda: test_env])
test_env = VecNormalize.load("./models/demo_vec_normalize.pkl", test_env)
test_env.training = False
test_env.norm_reward = False
model = SAC.load("./models/demo_model", env=test_env)

[WebUI] Listening at http://newton:38177


In [ ]:
# cleanup
test_env.close()

## Summary
By following the Gymnasium's environment standards, you have implemented and solved a more complicated driving scenario. Now you are able to utilize kdFlex's low-cost computational complexity to tackle increasingly harder problems. Good luck!